# 🚀 E-Commerce Big Data — Application & Inference Demo

**Step 7 of Big Data Architecture: Application & Model Serving Layer**

Tài liệu demo và kiểm thử tầng ứng dụng (Serving / Inference Layer). Tầng này cho phép các hệ thống downstream (Web API, CRM, Marketing Automation, Dashboard) truy xuất insights người dùng theo thời gian thực (real-time) hoặc xử lý hàng loạt (batch inference).

---

### Khả năng của Application Layer:
1. **Customer 360 Profile**: Tổng hợp chân dung khách hàng 360 độ từ hành vi quá khứ.
2. **Real-time Churn Prediction**: Dự báo xác suất rời bỏ và đề xuất chiến dịch can thiệp.
3. **Customer Segmentation**: Phân nhóm khách hàng vào 5 chân dung kinh doanh (Buyer Personas).
4. **Personalized Recommendations**: Gợi ý top sản phẩm theo sở thích danh mục & độ phổ biến.
5. **Batch Campaign Scoring**: Chấm điểm hàng loạt danh sách khách hàng để gửi khuyến mãi.

## 1. Setup & Khởi tạo Inference Engine

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import json

# Ensure project root is in path
PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.inference import (
    get_customer_360,
    predict_churn,
    get_segment,
    recommend_products,
    batch_inference
)
from src.storage import get_connection

print("✅ Inference Engine imported successfully!")

## 2. Lấy mẫu người dùng thực tế từ Database

In [ ]:
# Lấy danh sách 5 user có tương tác gần nhất trong hệ thống
conn = get_connection()
sample_users = pd.read_sql_query('''
    SELECT user_id, COUNT(*) as event_count, SUM(CASE WHEN event_type = 'purchase' THEN price ELSE 0 END) as total_spent
    FROM events
    GROUP BY user_id
    HAVING event_count > 5
    ORDER BY total_spent DESC
    LIMIT 5
''', conn)
conn.close()

print("Sample Active Users in Store:")
display(sample_users)

## 3. Demo: Customer 360 View (Hồ sơ khách hàng hợp nhất)

In [ ]:
target_user_id = int(sample_users.iloc[0]['user_id'])
customer_360 = get_customer_360(target_user_id)

print(f"==================================================")
print(f"   CUSTOMER 360 PROFILE FOR USER #{target_user_id}")
print(f"==================================================")
print(json.dumps(customer_360, indent=2, ensure_ascii=False))

## 4. Demo: Churn Risk Assessment (Dự báo nguy cơ rời bỏ)

In [ ]:
churn_result = predict_churn(target_user_id)

print(f"Target User ID: {target_user_id}")
print(f"Churn Status: {'CHURNED' if churn_result['is_churn'] else 'ACTIVE / RETAINED'}")
print(f"Churn Probability: {churn_result['churn_probability']:.2%}")
print(f"Risk Level: {churn_result['risk_level']}")
print(f"Action Strategy: {churn_result['suggested_action']}")
print(f"Key Drivers: {', '.join(churn_result['risk_drivers']) if churn_result['risk_drivers'] else 'None'}")

## 5. Demo: Customer Segmentation & Business Persona

In [ ]:
segment_result = get_segment(target_user_id)

print(f"Target User ID: {target_user_id}")
print(f"Assigned Cluster ID: {segment_result['cluster_id']}")
print(f"Persona: {segment_result['persona_name']}")
print(f"Recommended Marketing Campaign: {segment_result['marketing_strategy']}")

## 6. Demo: Personalized Recommendations

In [ ]:
recs = recommend_products(target_user_id, top_n=5)
recs_df = pd.DataFrame(recs)

print(f"Top 5 Recommended Products for User #{target_user_id}:")
display(recs_df)

## 7. Demo: Batch Inference cho Đội ngũ Marketing

In [ ]:
# Lấy 10 user để chấm điểm hàng loạt
conn = get_connection()
user_list = [r[0] for r in conn.execute("SELECT DISTINCT user_id FROM events LIMIT 10;").fetchall()]
conn.close()

batch_df = batch_inference(user_list)
print("Batch Scoring Results (Campaign Targeting):")
display(batch_df)

# Xuất file danh sách khách hàng có nguy cơ rời bỏ cao
high_risk = batch_df[batch_df['risk_level'] == 'HIGH']
print(f"\nSố khách hàng nguy cơ cao cần chăm sóc ngay: {len(high_risk)}")